# D1 · Check of the ViT-L/14 set-up (about 10 minutes)

In G3b, TANL with ViT-L/14 reaches Four-OOD FPR95 14.79, while TANL's paper reports 9.52 for ViT-L/14
(iNaturalist 0.29, SUN 3.42, Places 20.79, Textures 13.57). With ViT-B/16 the same code reproduces TANL (9.89 vs
9.81). This notebook tells a feature or text-bank problem apart from a TANL-specific one. It runs the two
methods without memory, NegLabel and MCM, whose ViT-L/14 numbers are published, on both backbones' caches
(one stream order is enough: they give the same result for every order), plus TANL with its released script's
settings (`official_sh`) on ViT-L/14.

* If NegLabel and MCM land near their published ViT-L/14 numbers, the L/14 features and text bank are right and
  the gap is TANL's own (the paper then reports our reproduction as it is).
* If they are far off on L/14 but not on B/16, the L/14 cache or text bank is the problem.

**Output:** `MyDrive/ReNeg/transfers/d1_vitl14_check/d1_runs.csv` (attach it).

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ctx = start("D1_vitl14_check", need_gpu=False)
log = ctx.log
import json, time
import numpy as np, pandas as pd, torch
from oodlab.report import find_cache
from oodlab.runner import Runner
from oodlab.methods import TANL, TANLConfig
try:
    from oodlab.methods import MCM, MCMConfig, NegLabel, NegLabelConfig
except ImportError:                                                  # the offline test stub of oodlab
    MCM = NegLabel = None
STUB = NegLabel is None or not hasattr(TANL, "selected_T")
from reneg.transport import l2n
OUT = f"{DRIVE_ROOT}/transfers/d1_vitl14_check"
os.makedirs(OUT, exist_ok=True)
runs_path = os.path.join(OUT, "d1_runs.csv")
# published Four-OOD FPR95 (iNaturalist, SUN, Places, Textures). NegLabel and MCM: ID positive (their papers);
# TANL: OpenOOD's convention (OOD positive). NegLabel L/14 as listed in TANL's Table A12.
PUB = {("neglabel", "ViT-B/16"): ([1.91, 20.53, 35.59, 43.56], "fpr95_idpos"),
       ("neglabel", "ViT-L/14"): ([1.77, 22.33, 32.22, 42.92], "fpr95_idpos"),
       ("mcm", "ViT-B/16"): ([30.91, 37.59, 44.69, 57.77], "fpr95_idpos"),
       ("mcm", "ViT-L/14"): ([28.38, 29.00, 35.42, 59.88], "fpr95_idpos"),
       ("tanl", "ViT-L/14"): ([0.29, 3.42, 20.79, 13.57], "fpr95"),
       ("tanl_official_sh", "ViT-L/14"): ([0.29, 3.42, 20.79, 13.57], "fpr95")}
FOUR = ["inaturalist", "sun", "places", "textures_all"]
print("device:", ctx.device)

## Run (saved after every method; re-running skips what is done)

In [ ]:
done = pd.read_csv(runs_path) if os.path.isfile(runs_path) else pd.DataFrame()
info = {}
for bb in (() if STUB else ("ViT-B/16", "ViT-L/14")):
    try:
        cache = find_cache(ctx, bb)
    except Exception as e:                                           # noqa: BLE001
        print(bb, "cache not found:", e)
        continue
    runner = Runner(cache, device=ctx.device, logger=log)
    if runner.missing("four_ood"):
        print(bb, "Four-OOD sets missing:", runner.missing("four_ood"))
        continue
    bank = cache.load_textbank()
    v = runner.get("imagenet_val")
    acc = float(((l2n(v.feats.float()) @ l2n(bank.id_text.float()).T).argmax(1).numpy() == np.asarray(v.labels)).mean() * 100)
    info[bb] = {"zero_shot_top1_val5k": round(acc, 2), "dim": int(bank.id_text.shape[1]),
                "corpus": int(bank.corpus_text.shape[0]), "n_selected": int(bank.n_selected),
                **{k: bank.meta.get(k) for k in ("backbone", "mining_backbone", "prompt", "noise_seed")}}
    print(bb, info[bb])
    neg = bank.corpus_text[: int(bank.n_selected)]
    todo = {"neglabel": lambda: NegLabel(bank.id_text, neg, NegLabelConfig(), device=ctx.device),
            "mcm": lambda: MCM(bank.id_text_simple, MCMConfig.paper(), device=ctx.device)}   # "a photo of a {}."
    if bb == "ViT-L/14":
        todo["tanl_official_sh"] = lambda: TANL(bank.id_text, bank.corpus_text, bank.noise_feats,
                                                TANLConfig.official_sh(), device=ctx.device,
                                                n_neglabel=int(bank.n_selected))
    for name, make in todo.items():
        if len(done) and ((done["label"] == name) & (done["backbone"] == bb)).any():
            print("done already:", name, bb)
            continue
        t0 = time.time()
        df = runner.evaluate(make(), "four_ood", seeds=[0], config=name)
        df["label"], df["backbone"] = name, bb
        done = pd.concat([done, df], ignore_index=True)
        done.to_csv(runs_path, index=False)
        print(f"{name} {bb}: {(time.time() - t0) / 60:.1f} min")
json.dump(info, open(os.path.join(OUT, "d1_info.json"), "w"), indent=1)
if STUB:
    print("test stub of oodlab: nothing to run")

## Ours against the published numbers, then attach the files

In [ ]:
df = pd.read_csv(runs_path) if os.path.isfile(runs_path) else pd.DataFrame(columns=["label", "backbone", "dataset"])
g3 = f"{DRIVE_ROOT}/transfers/g3_final_l14/g3_runs.csv"           # TANL (paper settings) from G3b, for reference
if os.path.isfile(g3):
    t = pd.read_csv(g3)
    t = t[(t.label == "tanl") & (t.protocol == "four_ood") & (t.seed == 0)].copy()
    t["backbone"] = "ViT-L/14"
    df = pd.concat([df, t], ignore_index=True)
rows = []
for (name, bb), g in df.groupby(["label", "backbone"]):
    g = g.set_index("dataset")
    pub, col = PUB.get((name, bb), (None, "fpr95"))
    ours = [float(g.loc[d, col]) if d in g.index else float("nan") for d in FOUR]
    row = {"method": name, "backbone": bb, "convention": "ID positive" if col == "fpr95_idpos" else "OOD positive",
           **{d: round(x, 2) for d, x in zip(FOUR, ours)}, "mean": round(float(np.mean(ours)), 2)}
    if pub:
        row["published mean"] = round(float(np.mean(pub)), 2)
        row["gap"] = round(row["mean"] - row["published mean"], 2)
    rows.append(row)
pd.set_option("display.width", 220)
print(pd.DataFrame(rows).to_string(index=False))
print("\nresult files:")
for f in (runs_path, os.path.join(OUT, "d1_info.json")):
    if os.path.isfile(f):
        print("  ", f)
finish(ctx, {"rows": int(len(df))})

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")